# Component declarations and connected programs

Native counterpart of every cell in the locked v0.7.0 infrastructure tutorial. Internal FunctionRecord and ComponentProgramBuilder own logical declarations. The executable extension uses circuit.append(op).

In [ ]:
import numpy as np
import torch
from IPython.display import SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.register_component_tutorial import (
    WordSwap,
    bit_swap_program,
    cnot_definition,
    modular_exponentiation_program,
    reallocation_declaration,
)
from flagquantum.compiler.registers.component_bindings import RegisterBindingError
from flagquantum.compiler.registers.component_builder import ComponentProgramBuilder
from flagquantum.core.ir.program import FunctionRecord, ProgramRecord
from flagquantum.core.ir.program.register_types import quantum_bit_type
from flagquantum.core.ir.program.registers import Register, RegisterSide, RegisterType
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import render_call_graph_svg
from flagquantum.drawer.component_graph import export_component_graph_dot


def show_declaration(function: FunctionRecord) -> None:
    signature = Signature(function.registers or ())
    return display(SVG(field_signature_svg(((function.name, signature, None),))))


def show_graph(record: ProgramRecord, entry: str) -> None:
    return display(
        SVG(render_call_graph_svg(export_component_graph_dot(record, entry)))
    )


The tutorial separates named declarations, connected decompositions and explicit numerical actions. A declaration alone does not imply an identity operation.

## Declarations

In [ ]:
factoring_declaration = FunctionRecord("FactoringAlgorithm", (), registers=())


A connected Program contains component definitions and SSA calls.

In [ ]:
cnot = cnot_definition()


Each port declares its name, width and direction. The default direction is THRU.

In [ ]:
control = Register("control", quantum_bit_type())
control


Finite typed register signatures are checked independently of component bodies.

In [ ]:
assert {register.name for register in Signature(cnot.registers)} == {
    "control",
    "target",
}


The declaration can be drawn before supplying its implementation.

In [ ]:
show_declaration(cnot)


The declared CNOT has no body, exactly as in the source tutorial. The final executable extension explicitly supplies CX.

## Decompositions

In [ ]:
swap_declaration = FunctionRecord(
    "SwapTwoBits", (), registers=tuple(Signature.build(x=1, y=1))
)


The two-bit swap has x and y ports.

In [ ]:
assert len(swap_declaration.registers) == 2


The component builder records calls in a reusable function.

In [ ]:
builder = ComponentProgramBuilder(Signature.build(x=1, y=1), (cnot,))


Three CNOT calls alternate control and target. Every call returns fresh handles.

In [ ]:
swap = bit_swap_program()
assert [op.callee for op in swap.functions[0].blocks[0].operations] == ["CNOT"] * 3


The stored return bindings preserve x and y order.

In [ ]:
show_declaration(swap.functions[0])


Drawing the declaration shows a single box. Drawing its body shows the connected calls.

In [ ]:
show_graph(swap, "SwapTwoBits")


The same builder constructs a standalone connected program.

In [ ]:
standalone_swap = bit_swap_program()
show_graph(standalone_swap, "SwapTwoBits")


## Linear ownership

Each quantum handle must be consumed once. Invalid calls below are deliberate examples.

In [ ]:
builder = ComponentProgramBuilder(Signature.build(x=1, y=1), (cnot,))
x, y = builder.inputs["x"], builder.inputs["y"]
try:
    builder.call_dict("CNOT", {"control": x, "target": x})
except RegisterBindingError as error:
    print("A linear variable cannot occupy both control and target.", error)
else:
    raise AssertionError("Repeated ownership was accepted")


In [ ]:
builder = ComponentProgramBuilder(Signature.build(x=1, y=1), (cnot,))
x, y = builder.inputs["x"], builder.inputs["y"]
outputs = builder.call_dict("CNOT", {"control": x, "target": y})
try:
    builder.call_dict("CNOT", {"control": x, "target": y})
except RegisterBindingError as error:
    print("Consumed input handles cannot be used again.", error)
else:
    raise AssertionError("Stale ownership was accepted")


In [ ]:
builder = ComponentProgramBuilder(Signature.build(x=1, y=1), (cnot,))
outputs = builder.call_dict(
    "CNOT", {"control": builder.inputs["x"], "target": builder.inputs["y"]}
)
outputs = builder.call_dict("CNOT", outputs)
try:
    builder.finalize({"x": outputs["control"]})
except (RegisterBindingError, ValueError) as error:
    print("Every live output must be returned.", error)
else:
    raise AssertionError("An unreturned live value was accepted")


RegisterBinding handles are allocated by the builder. Applications pass returned handles onward.

## Wide words and shaped arrays

In [ ]:
word_swap = WordSwap(5)
word_swap.signature


The immutable word operation is parameterized by width.

In [ ]:
assert WordSwap(5) == WordSwap(5)
assert WordSwap(5) != WordSwap(6)


In [ ]:
opaque_builder = ComponentProgramBuilder(word_swap.signature)
x_word = opaque_builder.inputs["x"]


A word handle is atomic until split.

In [ ]:
show_declaration(word_swap.declaration)


A wide declaration records width without constructing individual bit operations.

In [ ]:
show_declaration(WordSwap(10_000).declaration)


Indexing an opaque word directly is a deliberate type error.

In [ ]:
try:
    x_word[0]
except TypeError as error:
    print("An opaque word requires an explicit split.", error)
else:
    raise AssertionError("Opaque word was implicitly indexed")


Array shape describes multiple word handles, independently of each word width.

In [ ]:
array_register = Register("arr", RegisterType("opaque", 32), shape=(3, 3))
assert array_register.total_bits() == 288
print("total bits:", array_register.total_bits())


A shaped bit register has one independently addressable handle per element.

In [ ]:
array_swap = WordSwap(4).decompose(shaped=True)


The shaped version constructs one swap call per element.

In [ ]:
show_graph(array_swap, "SwapManyBits")


The signature shows each shaped element.

In [ ]:
show_declaration(array_swap.functions[0])


Explicit split and join preserve a compact external word signature.

In [ ]:
word_graph = WordSwap(5).decompose()


In [ ]:
show_graph(word_graph, "Swap")


The same connected program supports a musical score view.

In [ ]:
from flagquantum.drawer.component_score import component_register_score
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score

score = component_register_score(word_graph, "Swap")
figure, axes = draw_register_score(register_score_data(score))
figure.set_figwidth(9)


## Input and output ownership

In [ ]:
assert Register("ctrl", quantum_bit_type()).side == RegisterSide.THRU


LEFT inputs and RIGHT outputs can describe preparation and projection. Their numerical meaning requires an explicit supplier.

In [ ]:
reallocation = reallocation_declaration()
show_declaration(reallocation)


The following declared leaves use explicit tensors. ZeroEffect is a projection in this calculation.

In [ ]:
from collections.abc import Mapping

from flagquantum.simulation.tensor_network.component_dense import (
    contract_component_tensors,
)
from flagquantum.simulation.tensor_network.models import TensorNetworkNode
from flagquantum.simulation.tensor_network.register_tensor_binding import (
    bind_register_tensor,
)

zero = FunctionRecord(
    "ZeroState",
    (),
    registers=(Register("q", quantum_bit_type(), side=RegisterSide.RIGHT),),
)
show_declaration(zero)
zero_builder = ComponentProgramBuilder(Signature(zero.registers), (zero,), name="zero")
zero_outputs = zero_builder.call_dict("ZeroState", {})
zero_program = zero_builder.finalize(zero_outputs)


def state_effect_tensor(
    function: FunctionRecord,
    incoming: Mapping[str, tuple[tuple[int, ...], ...]],
    outgoing: Mapping[str, tuple[tuple[int, ...], ...]],
) -> tuple[TensorNetworkNode, ...]:
    data = {
        "ZeroState": torch.tensor([1, 0], dtype=torch.complex128),
        "PlusState": torch.ones(2, dtype=torch.complex128) / np.sqrt(2),
        "ZeroEffect": torch.tensor([1, 0], dtype=torch.complex128),
    }[function.name]
    return (
        bind_register_tensor(data, Signature(function.registers), incoming, outgoing),
    )


zero_tensor = contract_component_tensors(
    zero_program,
    (0,),
    state_effect_tensor,
    torch.empty((), dtype=torch.complex128),
    entry="zero",
)
assert torch.equal(zero_tensor, torch.tensor([1, 0], dtype=torch.complex128))
zero_tensor


In [ ]:
plus = FunctionRecord("PlusState", (), registers=zero.registers)
effect = FunctionRecord(
    "ZeroEffect",
    (),
    registers=(Register("q", quantum_bit_type(), side=RegisterSide.LEFT),),
)
projection_builder = ComponentProgramBuilder(
    Signature(()), (plus, effect), name="projection"
)
projection_builder.call_dict(
    "ZeroEffect", projection_builder.call_dict("PlusState", {})
)
projection = projection_builder.finalize({})
show_graph(projection, "projection")
projection_amplitude = contract_component_tensors(
    projection,
    (0, 1),
    state_effect_tensor,
    torch.empty((), dtype=torch.complex128),
    entry="projection",
)
projection_probability = projection_amplitude**2
assert abs(projection_probability.item() - 0.5) < 2e-16
projection_probability


## Modular exponentiation

In [ ]:
from flagquantum.algorithms.primitives.modular_exponentiation import (
    ModularExponentiation,
)

mod_exp = ModularExponentiation(8, 13 * 17, 3, 1024)
show_declaration(
    FunctionRecord("ModExp", (), registers=tuple(mod_exp.source_signature))
)


The connected decomposition initializes one output word, splits the exponent and calls controlled multipliers from the least significant bit upward.

In [ ]:
import inspect

print(inspect.getsource(modular_exponentiation_program))


The graph below is the actual connected one-level decomposition for exponent width 3 and output width 1024. It is drawn without expanding the multiplier leaves or materializing a 1024-qubit state. The source code excerpt omits initial modular reduction, while its actual method includes it; the native function follows the executed method.

In [ ]:
mod_exp_graph = modular_exponentiation_program(mod_exp)
show_graph(mod_exp_graph, "ModExp")


## Executable extension

The source tutorial stops at declared CNOT leaves. This extension explicitly supplies CX, binds both words and checks a complete complex state with an entangled spectator. The five-bit graph emits fifteen CX gates, uses no work bits and has no measurement branches. The original tutorial supplies no QEC resource model.

In [ ]:
import flagquantum as fq

state = torch.arange(2048, dtype=torch.float64)
state = torch.complex(state + 1, state.flip(0) + 3)
state /= state.norm()
circuit = fq.Circuit(11, inputs=state, dtype=torch.complex128)
circuit.append(WordSwap(5).make_on(tuple(range(5)), tuple(range(5, 10))))
expected = state.reshape(32, 32, 2).transpose(0, 1).flatten()
torch.testing.assert_close(circuit.state().flatten(), expected, rtol=0, atol=0)
assert len(circuit.to_ir().instructions) == 15
